# Testing on test dataset

In [1]:
import torch
print(torch.cuda.get_device_name(0))
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

NVIDIA GeForce RTX 2060
VRAM: 6.44 GB


In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import torch

model_name = "mistralai/Mistral-7B-Instruct-v0.2"

train_value = 2
adapter_dir = f"models/fine_tuned_model/train{train_value}"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    offload_folder="offload",
)

model = PeftModel.from_pretrained(base_model, adapter_dir)
model.eval()

tokenizer = AutoTokenizer.from_pretrained(adapter_dir, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token


c:\Users\salir\Desktop\Oulu\NLP\env310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 3/3 [00:47<00:00, 15.80s/it]


In [3]:
from datasets import load_dataset

raw_test = load_dataset(
    'json',
    data_files='data/train/test/val/test_instructions.jsonl',
    split='train'
)

In [ ]:
def build_prompt(sample):
    if train_value == 2:
        system = "You are a helpful assistant that answers questions about food and health relationships based on scientific evidence. Always cite your sources with page numbers."
    elif train_value == 3:
        system = "You are a helpful assistant that answers questions about food and health relationships based on scientific evidence. Always cite your sources with page numbers.  Offer detailed, discursive answers that clearly relate to the given sources."
    else:
        system = "You are a helpful assistant that answers questions about food and health relationships based on scientific evidence. Always cite your sources with page numbers. IMPORTANT: Rely primarily on the knowledge provided in the prompt or context. Only use your own internal knowledge when the provided information is insufficient. Offer detailed, discursive answers that clearly relate to the given sources."
    
    instruction = sample["instruction"]
    input_text = sample.get("input", "").strip()

    if input_text:
        user_message = f"{instruction}\n\nContext: {input_text}"
    else:
        user_message = instruction

    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user_message},
    ]

    return tokenizer.apply_chat_template(messages, tokenize=False)

In [4]:
def generate_answer(prompt, max_new_tokens=256):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return decoded.split("assistant")[-1].strip()

In [ ]:
predictions = []

for sample in raw_test:
    prompt = build_prompt(sample)
    answer = generate_answer(prompt)

    predictions.append({
        "instruction": sample["instruction"],
        "input": sample.get("input", ""),
        "expected_output": sample["output"],
        "model_output": answer
    })

In [7]:
import json
out_path = f'models/fine_tuned_model/train{train_value}/pred/test_predictions.jsonl'

with open(out_path, "w") as f:
    for row in predictions:
        f.write(json.dumps(row) + "\n")

print("Saved predictions to:", out_path)

Saved predictions to: models/fine_tuned_model/train3/pred/test_predictions.jsonl
